<a href="https://colab.research.google.com/github/AUCB21/DataEngineering/blob/main/TP1_AugustoContreras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env
%pip install deltalake

import requests
import pandas as pd
import pathlib
import datetime
import time
import os
from deltalake import DeltaTable, write_deltalake
from typing import Optional, Dict, Any
from dotenv import load_dotenv

load_dotenv('.env')

api_key = os.getenv('VANTAGE_API_KEY')

if not api_key:
    print("API key not found in .env file")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.7/38.7 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 40.4 MB/s eta 0:00:00
API key not found in .env file


Phase 1: Fetch data - Endpoints:


*   https://www.alphavantage.co/query?
*   Parameters:
    *    Required:
          * apikey
          * function
          * symbol
          * interval ( 1min, 5min, 15min, 30min, 60min)
    *   Optional:
        * outputsize (compact, full)
        * datatype (json, csv)


**Examples**

The API will return the most recent 100 intraday OHLCV bars by default when the outputsize parameter is not set
https://www.alphavantage.co/query?function=TIME_SERIES_INTRADAY&symbol=IBM&interval=5min&apikey=demo

Query the most recent 30 days of intraday data by setting outputsize=full
https://www.alphavantage.co/query?function=TIME_SERIES_INTRADAY&symbol=IBM&interval=5min&outputsize=full&apikey=demo

Query intraday data for a full month in history (e.g., 2009-01). Any month in the last 20+ years (since 2000-01) is supported
https://www.alphavantage.co/query?function=TIME_SERIES_INTRADAY&symbol=IBM&interval=5min&month=2009-01&outputsize=full&apikey=demo

In [ ]:
from pprint import pprint
# pandas.json_normalize() - Pending


class APIExtractor:
  """
  Clase encargada de extraccion (fetches) de datos al endpoint
  """
  BASE_URL = "https://www.alphavantage.co/query"

  def __init__(self, apikey: str, timeout: int):
    self.apikey = apikey
    self.session = requests.Session()
    self.timeout = timeout

  def _fetch_data(self, params: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """
    Ejecuta calls HTTP con parametros dados
    parametros: any given in child, timeout in seconds
    """
    params["apikey"] = self.apikey
    print(f"Fetching data for {params['symbol']} with function {params['function']}...")

    try:
      response = self.session.get(self.BASE_URL, params=params)
      response.raise_for_status()
      time.sleep(self.timeout)

      if response.status_code == 200:
        print(f"Successfully fetched data for {params['symbol']}")
        pprint(response.json())
        return response.json()
      else:
        print(f"Error fetching data: {response.status_code}")
        return None
    except Exception as e:
      print(f"Error fetching data: {e}")
      return None

  def get_incremental_data(self, symbol: str, function: str = "TIME_SERIES_DAILY", outputsize: str = "compact") -> pd.DataFrame:
    """
    Consigna 1: Extraccion Incremental de data via endpoint
    parametros: symbol, function, outputsize
    """
    paramsInc = {
        "function": function,
        "symbol": symbol,
        "outputsize": outputsize # FULL solo permitido en endpoints premium
    }

    raw_data = self._fetch_data(params=paramsInc)
    if not raw_data:
      return pd.DataFrame()

    # Localizar la llave que contiene la serie (dinámica según la función)
    time_series_key = next((key for key in raw_data.keys() if "Time Series" in key), None)


    if time_series_key:
        # Transformación a DataFrame
        df = pd.DataFrame.from_dict(raw_data[time_series_key], orient='index')
        df.index.name = 'date'
        df = df.reset_index()
        # Limpieza básica: nombres de columnas legibles
        df.columns = [col.split(". ")[-1].replace(" ", "_") for col in df.columns]
        df['symbol'] = symbol
        return df

    return pd.DataFrame()

  def get_static_data(self, symbol: str) -> pd.DataFrame:
    """
    Consigna 2: Extraccion FULL del endpoint (function=OVERVIEW en API elegida)
    parametros: symbol
    """
    paramsFull = {
        "function": "OVERVIEW",
        "symbol": symbol
    }
    data = self._fetch_data(params=paramsFull)

    if data:
      # Conversion de diccionario a DataFrame y agregado de metadata para control
      df = pd.DataFrame([data])
      df["extraction_timestamp"] = datetime.datetime.now()
      return df
    return pd.DataFrame()


Pasos a seguir:


1.   Inicializacion de instancia:
```
extractor = APIExtractor(api_key=api_key)
```


2.   Extraccion de informacion:
* Incremental:
```
df_info = extractor.get_static_data("SMBL")
```
* Estatico (FULL):
```
df_prices = extractor.get_incremental_data("SMBL")
```







In [ ]:
extractor = APIExtractor(apikey=api_key, timeout=10)
df_info = extractor.get_static_data("IBM")
df_prices = extractor.get_incremental_data("NVDA")

Fetching data for IBM with function OVERVIEW...
Successfully fetched data for IBM
{'Error Message': 'the parameter apikey is invalid or missing. Please claim '
                  'your free API key on '
                  '(https://www.alphavantage.co/support/#api-key). It should '
                  'take less than 20 seconds.'}
Fetching data for NVDA with function TIME_SERIES_DAILY...
Successfully fetched data for NVDA
{'Error Message': 'the parameter apikey is invalid or missing. Please claim '
                  'your free API key on '
                  '(https://www.alphavantage.co/support/#api-key). It should '
                  'take less than 20 seconds.'}


In [ ]:
def transform_static_to_df(self, raw_data: dict) -> pd.DataFrame:
    """
    Convierte el JSON de metadatos en un DataFrame de una fila
    """
    if (not raw_data) or ("Symbol" not in raw_data):
        return pd.DataFrame()

    df = pd.DataFrame([raw_data])

    # Transformaciones: Convertir strings a tipos numericos para emprolijado
    # basados en respuestas hechas en Test
    numeric_cols = [
        '200DayMovingAverage', '50DayMovingAverage', 'MarketCapitalization',
        'EBITDA', 'PERatio', 'EPS', 'DividendYield'
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

In [ ]:
def transform_incremental_to_df(self, raw_data: dict) -> pd.DataFrame:
    """
    Transforma la serie diaria, limpia nombres de columnas y agrega particiones correspondientes
    """
    if 'Time Series (Daily)' not in raw_data:
        return pd.DataFrame()

    # Carga diccionario dado en parametro
    df = pd.DataFrame.from_dict(raw_data['Time Series (Daily)'], orient='index')

    df.index.name = 'date'
    df = df.reset_index()

    # Limpieza de nombres: "1. open" -> "open", eliminado espacios en blanco " "
    df.columns = [col.split(". ")[-1].replace(" ", "_") for col in df.columns]

    # 4. Casteo de tipos str -> int
    df['date'] = pd.to_datetime(df['date'])
    numeric_cols = ['open', 'high', 'low', 'close', 'volume']
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

    # 5. Columnas de Particionado (Requisito Obligatorio)
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day

    # Agregamos el símbolo para identificar la data en el lake
    df['symbol'] = raw_data['Meta Data']['2. Symbol']

    return df

In [ ]:
def write_to_delta_lake(data):
  data = pandas.DataFrame(data)
  write_deltalake()